# GitHub Pull Requests 匯出成 JSONL

這個 Notebook 會抓取指定 GitHub 公開 repository 的 Pull Requests，並輸出成 JSONL。

每筆資料至少包含以下欄位：
- `url`
- `title`
- `pr_description`
- `is_closed`
- `is_approved`
- `comments`（陣列，含發文者 `poster`）
- `last_commit_gitdiff`

## 使用方式
1. 在下一個 cell 設定 `PULLS_URL` 或 `REPO`。
2. 若有 GitHub token，建議設定環境變數 `GITHUB_TOKEN` 以避免 rate limit。
3. 執行全部 cells。
4. 輸出檔案會在 `data/exports/*.jsonl`。

In [13]:
import json
import os
import re
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from urllib.parse import urlparse

from dotenv import load_dotenv
import requests

load_dotenv("../.env") 

# ===== 使用者可調整參數 =====
OWNER_REPO = "spring-projects/spring-ai"
STATE = "all"               # open | closed | all
MAX_PRS = None                # None 代表抓全部
OUTPUT_DIR = Path("exports")
SLEEP_SECONDS = 0.1 
RELEASE_TAG: Optional[str] = "v2.0.0-M1"  # 例如: "v2.0.0-M1"。若有值會只抓取該 release 之後的 PRs；設為 None 表示不篩選
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
if GITHUB_TOKEN:
    headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

OWNER, REPO_NAME = OWNER_REPO.split("/", 1)
BASE_API = f"https://api.github.com/repos/{OWNER_REPO}"

session = requests.Session()
session.headers.update(headers)

print(f"Target repo: {OWNER_REPO}")
print(f"Token set: {'yes' if GITHUB_TOKEN else 'no'}")


# 如果指定了 RELEASE_TAG，獲取該 release 的發布時間
RELEASE_DATE: Optional[str] = None
if RELEASE_TAG:
    try:
        def gh_get_temp(url: str, params: Optional[Dict] = None):
            """暫時的 gh_get 用於獲取 release 資訊"""
            resp = session.get(url, params=params, timeout=30)
            if resp.status_code == 403 and "rate limit" in resp.text.lower():
                raise RuntimeError(f"Rate limit exceeded")
            resp.raise_for_status()
            return resp.json()
        
        release_info = gh_get_temp(f"{BASE_API}/releases/tags/{RELEASE_TAG}")
        RELEASE_DATE = release_info.get("published_at") or release_info.get("created_at")
        print(f"Release '{RELEASE_TAG}' published at: {RELEASE_DATE}")
    except Exception as e:
        print(f"Warning: Failed to fetch release '{RELEASE_TAG}': {e}")

if RELEASE_DATE:
    print(f"Filtering PRs created after: {RELEASE_DATE}")

Target repo: spring-projects/spring-ai
Token set: yes
Release 'v2.0.0-M1' published at: 2025-12-11T21:52:49Z
Filtering PRs created after: 2025-12-11T21:52:49Z


In [ ]:
def gh_get(url: str, params: Optional[Dict] = None, accept: Optional[str] = None, raw_text: bool = False):
    """單次 GitHub API 請求。"""
    local_headers = {}
    if accept:
        local_headers["Accept"] = accept

    resp = session.get(url, params=params, headers=local_headers or None, timeout=30)

    if resp.status_code == 403 and "rate limit" in resp.text.lower():
        reset_at = resp.headers.get("X-RateLimit-Reset")
        raise RuntimeError(f"Rate limit exceeded. X-RateLimit-Reset={reset_at}")

    resp.raise_for_status()
    if raw_text:
        return resp.text
    return resp.json()


def fetch_paginated(url: str, params: Optional[Dict] = None, max_items: Optional[int] = None) -> List[Dict]:
    """抓取 GitHub 分頁資料。"""
    results: List[Dict] = []
    page = 1
    per_page = 100

    while True:
        q = dict(params or {})
        q.update({"page": page, "per_page": per_page})
        batch = gh_get(url, params=q)

        if not isinstance(batch, list) or len(batch) == 0:
            break

        for item in batch:
            results.append(item)
            if max_items is not None and len(results) >= max_items:
                return results

        if len(batch) < per_page:
            break

        page += 1
        if SLEEP_SECONDS > 0:
            time.sleep(SLEEP_SECONDS)

    return results


def compute_is_approved(reviews: List[Dict]) -> bool:
    """以每位 reviewer 最後狀態判斷是否仍存在有效 APPROVED。"""
    latest_state_by_user: Dict[str, str] = {}

    ordered_reviews = sorted(reviews, key=lambda r: r.get("submitted_at") or "")
    for rv in ordered_reviews:
        user = (rv.get("user") or {}).get("login")
        state = (rv.get("state") or "").upper()
        if user:
            latest_state_by_user[user] = state

    return any(state == "APPROVED" for state in latest_state_by_user.values())


def collect_comments(issue_comments: List[Dict], review_comments: List[Dict], reviews: List[Dict]) -> List[Dict]:
    """統一整理訊息格式，並標記 poster。"""
    comments: List[Dict] = []

    for c in issue_comments:
        comments.append({
            "type": "issue_comment",
            "poster": (c.get("user") or {}).get("login"),
            "created_at": c.get("created_at"),
            "url": c.get("html_url"),
            "body": c.get("body") or "",
        })

    for c in review_comments:
        comments.append({
            "type": "review_comment",
            "poster": (c.get("user") or {}).get("login"),
            "created_at": c.get("created_at"),
            "url": c.get("html_url"),
            "body": c.get("body") or "",
        })

    for rv in reviews:
        comments.append({
            "type": "review_event",
            "poster": (rv.get("user") or {}).get("login"),
            "created_at": rv.get("submitted_at"),
            "url": rv.get("html_url"),
            "state": rv.get("state"),
            "body": rv.get("body") or "",
        })

    comments.sort(key=lambda m: m.get("created_at") or "")
    return comments


def fetch_last_commit_diff(pr_number: int) -> str:
    commits_url = f"{BASE_API}/pulls/{pr_number}/commits"
    commits = fetch_paginated(commits_url)
    if not commits:
        return ""

    last_sha = commits[-1].get("sha")
    if not last_sha:
        return ""

    commit_api = f"{BASE_API}/commits/{last_sha}"
    return gh_get(commit_api, accept="application/vnd.github.v3.diff", raw_text=True)


def build_pr_record(pr: Dict) -> Dict:
    pr_number = pr["number"]

    issue_comments = fetch_paginated(f"{BASE_API}/issues/{pr_number}/comments")
    review_comments = fetch_paginated(f"{BASE_API}/pulls/{pr_number}/comments")
    reviews = fetch_paginated(f"{BASE_API}/pulls/{pr_number}/reviews")
    last_commit_gitdiff = fetch_last_commit_diff(pr_number)

    return {
        "url": pr.get("html_url"),
        "title": pr.get("title") or "",
        "pr_description": pr.get("body") or "",
        "is_closed": (pr.get("state") or "").lower() == "closed",
        "is_approved": compute_is_approved(reviews),
        "comments": collect_comments(issue_comments, review_comments, reviews),
        "last_commit_gitdiff": last_commit_gitdiff,
    }


def fetch_pull_requests(state: str = "all", max_prs: Optional[int] = None, since_date: Optional[str] = None):
    params = {"state": state, "sort": "created", "direction": "desc"}

    count = 0
    page = 1
    per_page = 100
    stop = False

    while not stop:
        q = dict(params)
        q.update({"page": page, "per_page": per_page})
        pulls = gh_get(f"{BASE_API}/pulls", params=q)

        if not isinstance(pulls, list) or len(pulls) == 0:
            break

        for pr in pulls:
            if since_date and (pr.get("created_at") or "") <= since_date:
                stop = True
                break

            count += 1
            print(f"[{count}] Processing PR #{pr['number']} - {pr.get('title', '')}")
            record = build_pr_record(pr)
            yield record
            if max_prs is not None and count >= max_prs:
                return
            if SLEEP_SECONDS > 0:
                time.sleep(SLEEP_SECONDS)

        if len(pulls) < per_page:
            break

        page += 1

In [16]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_name = f"{OWNER}_{REPO_NAME}_prs_{STATE}.jsonl"
out_path = OUTPUT_DIR / out_name
write_mode = "a" if out_path.exists() else "w"

count = 0
with out_path.open(write_mode, encoding="utf-8") as f:
    for record in fetch_pull_requests(state=STATE, max_prs=MAX_PRS, since_date=RELEASE_DATE):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        count += 1

print(f"Exported {count} pull requests")
print(f"JSONL path: {out_path}")

[1] Processing PR #5993 - GH-5987: fix OpenAI chat streaming latency without tools
[2] Processing PR #5992 - fix: DeepSeek auto-config should respect spring.ai.deepseek.chat.enabled
[3] Processing PR #5990 - Fix OpenAI auto-config when only sub-prefix api-key is set
[4] Processing PR #5979 - GH-5971: fix streaming tool-call observation context propagation
[5] Processing PR #5976 - OllamaChatOptions getOutputSchema() throws exception
[6] Processing PR #5975 - feat: allow custom modules in BeanOutputConverter JSON schema generation
[7] Processing PR #5973 - fix: Fix streaming tool call merge logic to use index field instead of id
[8] Processing PR #5969 - Deprecate SSE transports, set Streamable HTTP as default server protocol
[9] Processing PR #5968 - GH-5963: pass back `reasoning_content` in tool-calling for DeepSeek API
[10] Processing PR #5967 - WebFluxSseClientTransport: introduce SSE message endpoint validator
[11] Processing PR #5962 - Improve chat model observations
[12] Processi